# 特征选择：只留最有用的特征
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：11_特征工程 | 源文件：特征选择.py | 核心功能：过滤法、包装法、嵌入法

## 概述

特征选择从原始特征中选出最有用的子集。三大方法：过滤法（基于统计量筛选）、包装法（用模型评估特征子集）、嵌入法（模型训练中自动选择）。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 数学原理

### 1. 特征选择的目标

特征选择从原始 $d$ 个特征中选出子集 $S \subset \{1,\ldots,d\}$，$|S| = k < d$，使得模型在 $S$ 上的性能最优：

$$S^* = \arg\max_{S: |S|=k} J(S)$$

其中 $J(S)$ 是评价函数（如交叉验证准确率）。搜索所有子集的复杂度为 $O(2^d)$，需要启发式方法。

### 2. 方差阈值法（Filter）

最简单的过滤方法：移除方差低于阈值的特征。

$$\text{Var}(x_j) = \frac{1}{n}\sum_{i=1}^{n}(x_{ij} - \bar{x}_j)^2$$

方差接近 0 的特征（近似常量）对预测无贡献。对于二值特征，方差为：

$$\text{Var}(x_j) = p(1-p)$$

其中 $p$ 是取 1 的频率。

### 3. 单变量统计检验（SelectKBest）

**F 检验（分类）**：衡量特征与目标的线性依赖

$$F_j = \frac{\text{组间方差}}{\text{组内方差}} = \frac{\sum_c n_c(\bar{x}_{jc} - \bar{x}_j)^2 / (C-1)}{\sum_c \sum_{i \in c}(x_{ij} - \bar{x}_{jc})^2 / (n-C)}$$

$F_j$ 越大，特征 $j$ 对分类的区分能力越强。

**互信息**：衡量特征与目标的非线性依赖

$$I(X_j; Y) = \sum_{x,y} P(x,y) \log \frac{P(x,y)}{P(x)P(y)}$$

$I(X_j; Y) = 0$ 当且仅当 $X_j$ 与 $Y$ 独立。互信息能捕捉非线性关系。

### 4. 递归特征消除（RFE）

迭代算法：
1. 用所有特征训练模型
2. 移除权重绝对值最小的特征
3. 重复直到剩余 $k$ 个特征

$$j^* = \arg\min_{j \in S} |w_j|$$

对于树模型，用 `feature_importances_` 替代权重。

### 5. 基于模型的特征选择（SelectFromModel）

用带 `feature_importances_` 的模型（如随机森林）筛选：

$$S = \{j : \text{importance}_j > \text{threshold}\}$$

阈值可以是均值、中位数或自定义值。

### 6. L1 正则化（嵌入式方法）

Lasso 回归通过 L1 正则化自动进行特征选择：

$$\min_w \|y - Xw\|^2 + \lambda\|w\|_1$$

L1 正则化使部分权重精确为 0，实现自动特征选择。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `VarianceThreshold(threshold=0.1)` | 移除 $\text{Var}(x_j) < 0.1$ 的特征 |
| `SelectKBest(f_classif, k=5)` | 选 F 值最大的 5 个特征 |
| `mutual_info_classif(X, y)` | 计算 $I(X_j; Y)$ |
| `RFE(estimator, n_features_to_select=5)` | 递归消除，保留 5 个特征 |
| `SelectFromModel(rf, threshold="mean")` | 重要性 $> \bar{\text{importance}}$ 的特征 |
| `get_support(indices=True)` | 返回被选特征的索引集合 $S$ |

### 1. 方差阈值 (VarianceThreshold)

运行 1. 方差阈值 (VarianceThreshold) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.feature_selection import VarianceThreshold

print("=" * 60)
print("1. 方差阈值 (VarianceThreshold)")
print("=" * 60)

# 生成数据：10个特征，其中2个是近似常量（方差很小）
np.random.seed(42)
X_var = np.random.randn(200, 10)
X_var[:, 8] = 0.01  # 近似常量特征
X_var[:, 9] = 0.005  # 近似常量特征

print(f"原始特征数: {X_var.shape[1]}")
print(f"各特征方差: {np.round(X_var.var(axis=0), 4)}")

# 阈值设为0.1，方差低于此值的特征会被移除
selector_var = VarianceThreshold(threshold=0.1)
X_selected = selector_var.fit_transform(X_var)

print(f"方差阈值=0.1 后剩余特征数: {X_selected.shape[1]}")
print(f"被保留的特征索引: {selector_var.get_support(indices=True)}")
print(f"被移除的特征索引: {np.where(~selector_var.get_support())[0]}")

### 2. 单变量统计检验 (SelectKBest + f_classif)

运行 2. 单变量统计检验 (SelectKBest + f_classif) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

print("\n" + "=" * 60)
print("2. 单变量统计检验 (SelectKBest + f_classif)")
print("=" * 60)

X_cls, y_cls = make_classification(
    n_samples=300, n_features=15, n_informative=5,
    n_redundant=3, n_classes=2, random_state=42
)

# 选择K=5个最佳特征
selector_kbest = SelectKBest(score_func=f_classif, k=5)
X_kbest = selector_kbest.fit_transform(X_cls, y_cls)

print(f"原始特征数: {X_cls.shape[1]}")
print(f"选择K=5后特征数: {X_kbest.shape[1]}")
print(f"各特征的F统计量: {np.round(selector_kbest.scores_, 2)}")
print(f"被选中的特征索引: {selector_kbest.get_support(indices=True)}")
print(f"被选中特征的p值: {np.round(selector_kbest.pvalues_[selector_kbest.get_support()], 6)}")

### 3. 互信息 (mutual_info_classif)

运行 3. 互信息 (mutual_info_classif) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.feature_selection import mutual_info_classif

print("\n" + "=" * 60)
print("3. 互信息 (mutual_info_classif)")
print("=" * 60)

# 互信息能捕捉非线性关系，适合更复杂的特征依赖
mi_scores = mutual_info_classif(X_cls, y_cls, random_state=42)

print("各特征的互信息得分:")
for i, score in enumerate(mi_scores):
    print(f"  特征{i}: {score:.4f}")

# 选择互信息得分最高的5个特征
top5_idx = np.argsort(mi_scores)[::-1][:5]
print(f"\n互信息Top5特征索引: {top5_idx}")
print(f"互信息Top5得分: {mi_scores[top5_idx].round(4)}")

### 4. 递归特征消除 (RFE)

运行 4. 递归特征消除 (RFE) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

print("\n" + "=" * 60)
print("4. 递归特征消除 (RFE)")
print("=" * 60)

# RFE反复训练模型，每次移除权重最小的特征
estimator = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=estimator, n_features_to_select=5, step=1)
X_rfe = rfe.fit_transform(X_cls, y_cls)

print(f"原始特征数: {X_cls.shape[1]}")
print(f"RFE选择后特征数: {X_rfe.shape[1]}")
print(f"各特征排名 (1=最佳): {rfe.ranking_}")
print(f"被选中的特征索引: {rfe.get_support(indices=True)}")
print(f"被移除的特征索引: {np.where(~rfe.get_support())[0]}")

### 5. 基于模型的特征选择 (SelectFromModel)

运行 5. 基于模型的特征选择 (SelectFromModel) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

print("\n" + "=" * 60)
print("5. 基于模型的特征选择 (SelectFromModel)")
print("=" * 60)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_cls, y_cls)

print(f"随机森林feature_importances_: {np.round(rf.feature_importances_, 4)}")

# 使用中位数作为阈值
selector_sfm = SelectFromModel(rf, threshold="median")
X_sfm = selector_sfm.fit_transform(X_cls, y_cls)

print(f"原始特征数: {X_cls.shape[1]}")
print(f"SelectFromModel选择后特征数: {X_sfm.shape[1]}")
print(f"被选中的特征索引: {selector_sfm.get_support(indices=True)}")

# 使用绝对值阈值
selector_sfm2 = SelectFromModel(rf, threshold=0.05)
X_sfm2 = selector_sfm2.fit_transform(X_cls, y_cls)
print(f"\n绝对阈值0.05选择后特征数: {X_sfm2.shape[1]}")
print(f"被选中的特征索引: {selector_sfm2.get_support(indices=True)}")

### 综合对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("综合对比：各方法选出的特征集")
print("=" * 60)
print(f"SelectKBest (F检验): {sorted(selector_kbest.get_support(indices=True))}")
print(f"互信息Top5:          {sorted(top5_idx)}")
# --- 输出结果 ---
print(f"RFE:                 {sorted(rfe.get_support(indices=True))}")
print(f"SelectFromModel(中位):{sorted(selector_sfm.get_support(indices=True))}")
print(f"SelectFromModel(0.05):{sorted(selector_sfm2.get_support(indices=True))}")

## 关键代码解释

```python
from sklearn.feature_selection import SelectKBest, RFE, SelectFromModel
# 过滤法
SelectKBest(k=10)
# 包装法
RFE(estimator, n_features_to_select=5)
# 嵌入法
SelectFromModel(Lasso(alpha=0.1))
```

## 注意事项

1. 特征选择必须在训练集上做，不能用测试集信息
2. 包装法计算量大但效果通常最好
3. 嵌入法与正则化模型天然结合

## 总结与延伸

以上代码展示了 **特征选择** 的完整流程。

**解读要点**：
- 关注 **特征重要性** 排名和分布
- 对比特征选择前后的模型性能
- 观察特征交叉是否带来性能提升

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Boruta**：基于随机森林的全相关特征选择
- **SHAP-based 选择**：用 SHAP 值排序特征
- **特征选择 vs 降维**：前者保留原始特征，后者创建新特征
